## **Load in the model**

In [2]:
from pathlib import Path
import hydra
from hydra import compose, initialize
import torch

from FM_HiREF.model.hireflow import HiReFlow

try:
    initialize(version_base="1.3", config_path="configs", job_name="notebook_eval")
except ValueError:
    pass

check_point = Path("/home/amrinder/Documents/Github/Protein-FM-HiREF/weights/scope_weights.ckpt")

cfg = compose(config_name="config")
net = hydra.utils.instantiate(cfg.net)

model = HiReFlow.load_from_checkpoint(checkpoint_path=check_point, net=net,)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

HiReFlow(
  (net): FlowMatchingNetwork(
    (c_embedder): CondFeatureNet(
      (linear_c): Linear(in_features=128, out_features=128, bias=False)
      (transition_1): Transition(
        (linear): Linear(in_features=128, out_features=512, bias=False)
        (swiglu): SwiGLU()
        (linear_out): Linear(in_features=256, out_features=128, bias=False)
      )
      (transition_2): Transition(
        (linear): Linear(in_features=128, out_features=512, bias=False)
        (swiglu): SwiGLU()
        (linear_out): Linear(in_features=256, out_features=128, bias=False)
      )
    )
    (s_embedder): SingleFeatureNet(
      (s_linear): Linear(in_features=128, out_features=192, bias=False)
    )
    (p_embedder): PairFeatureNet(
      (p_ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (p_linear): Linear(in_features=63, out_features=128, bias=False)
      (c_pair_linear): Linear(in_features=128, out_features=128, bias=False)
      (c_pair_ln): LayerNorm((128,), eps=1e-05, el

### **Sample Protein**

In [24]:
batch = 4
n_res = 80
steps = 25

with torch.no_grad():
    x_1, trajectory = model.sample_protein(batch_size=batch, 
                                           num_res=n_res, 
                                           num_steps=steps,
                                           scale=cfg.data.scale,
                                           save_trajectory=True, 
                                           seed=cfg.seed)
    
    x_1, trajectory = x_1.cpu(), trajectory.cpu()

print(f"Tensor Shape: {x_1.shape} [B, N_RES, 3]")

Folding 4 proteins (80 res): 100%|██████████| 25/25 [00:34<00:00,  1.37s/it]

Tensor Shape: torch.Size([4, 80, 3]) [B, N_RES, 3]


### **View Protein**

In [25]:
import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import py3Dmol

import numpy as np

def tensor_to_pdb(coords):  
    coords_np = coords.detach().cpu().numpy()
    n_atoms = coords_np.shape[0]
    atoms = struc.AtomArray(n_atoms)
    
    # Fill in the AtomArray object
    atoms.coord = coords_np
    atoms.atom_name = np.full(n_atoms, "CA")
    atoms.res_name = np.full(n_atoms, "ALA")
    atoms.res_id = np.arange(1, n_atoms + 1)
    atoms.element = np.full(n_atoms, "C")
    atoms.chain_id = np.full(n_atoms, "A")
    
    pdb_file = pdb.PDBFile()
    pdb_file.set_structure(atoms)
    
    return str(pdb_file)

# Select structure to view
to_view =  x_1[0]
pdb_string = tensor_to_pdb(to_view)

# View
view = py3Dmol.view(width=800, height=400)
view.addModel(pdb_string, 'pdb')

view.setStyle(
    {'model': 0},
    {
        "sphere": {
            "radius": 0.8,
            "colorscheme": {
                "prop": "resi",
                "gradient": "sinebow",
                "min": 1,
                "max": 100
            }
        },
        "cartoon": {
            "style": "trace",
            "colorscheme": {
                "prop": "resi",
                "gradient": "sinebow",
                "min": 1,
                "max": 100
            }
        }
    }
)

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.